In [ ]:
#Replace string with int ratings
bug_data["severity"] = bug_data["severity"].replace({"Low": 1, "Medium": 2, "High": 3, "Critical": 4})
bug_data["priority"] = bug_data["priority"].replace({"Low": 1, "Normal": 2, "High": 3, "Urgent": 4})

#Recode severity
def calculate_severity(row):
    score = 0
    # Environment
    if row["environment"] == "Production":
        score += 2
    elif row["environment"] == "Staging":
        score += 1
    # Bug Category
    if row["bug_category"] == "Security Vulnerability":
        score += 4
    elif row["bug_category"] in [
        "Memory Leak",
        "Database Bug",
        "Concurrency Bug",
        "Authentication Bug",
        "Authorization Bug"
    ]:
        score += 3
    elif row["bug_category"] in [
        "Backend Logic Bug",
        "Performance Bug",
        "API Bug",
        "Deployment Bug",
        "Cloud Configuration Bug"
    ]:
        score += 2
    else:
        score += 1
    # Error Code
    if row["error_code"] >= 500:
        score += 2
    elif row["error_code"] >= 400:
        score += 1
    return score

#Apply recode
bug_data["severity_score"] = bug_data.apply(
    calculate_severity,
    axis=1
)

def severity_class(score):
    if score <= 2:
        return 1      # Low
    elif score <= 4:
        return 2      # Medium
    elif score <= 6:
        return 3      # High
    else:
        return 4      # Critical

bug_data["severity"] = bug_data["severity_score"].apply(severity_class)

#Recode priority
def priority_class(row):
    score = row["severity"]
    if row["environment"] == "Production":
        score += 1
    if score >= 5:
        return 4      # P1 Critical
    elif score == 4:
        return 3      # P2 High
    elif score == 3:
        return 2      # P3 Normal
    else:
        return 1      # P4 Low

#Apply recode
bug_data["priority"] = bug_data.apply(
    priority_class,
    axis=1
)

#Export new file
bug_data_corrected = bug_data
bug_data_corrected.to_csv(Data_Path + "bug_dataset_with_priority_and_severity_corrected.csv", index=False)

In [ ]:
#Number of bugs
print("Number of bugs: ", len(bug_data))

#Number of errors per error_code
print("Number of errors per error_code: \n", bug_data["error_code"].value_counts())

#Number of errors per severity
print("Number of errors per severity: \n", bug_data["severity"].value_counts())

#Number of errors per priority rating
print("Number of errors per priority rating: \n", bug_data["priority"].value_counts())

#Number of errors per environment
print("Number of errors per environment: \n", bug_data["environment"].value_counts())

In [ ]:
#Severity by bug category
print("Priority by bug category: \n", bug_data.groupby("bug_category")["priority"].mean())

#Priority by bug category
print("Severity by bug category: \n", bug_data.groupby("bug_category")["severity"].mean())

#Severity by environment
environment_severity = bug_data.groupby("environment")["severity"].agg
print("Severity by environment: \n", bug_data.groupby("environment")["severity"].mean())

#Priority by environment
print("Priority by environment: \n", bug_data.groupby("environment")["priority"].mean())

#Severity by bug domain
print("Severity by bug domain: \n", bug_data.groupby("bug_domain")["severity"].mean())

#Priority by bug domain
print("Priority by bug domain: \n", bug_data.groupby("bug_domain")["priority"].mean())

In [ ]:
#Bar Plot of Severity based on Bug Category

severity_by_domain = bug_data.groupby("bug_category")["severity"].mean().sort_values(ascending=True)

plt.figure(figsize=(12, 6))
sns.barplot(x=severity_by_domain.index, y=severity_by_domain.values, palette='viridis')
plt.title('Mean Severity Rating by Bug Category')
plt.xlabel('Bug Domain')
plt.ylabel('Mean Severity Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Bar Plot of Severity by environment

severity_by_environment = bug_data.groupby("environment")["severity"].median().sort_values(ascending = True)

plt.figure(figsize=(12, 6))
sns.barplot(x=severity_by_environment.index, y=severity_by_environment.values, palette='viridis')
plt.title('Median Severity Rating by Environment')
plt.xlabel('Environment')
plt.ylabel('Median Severity Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Bar Plot of Priority by environment

priority_by_environment = bug_data.groupby("environment")["priority"].median().sort_values(ascending = True)

plt.figure(figsize=(12, 6))
sns.barplot(x=priority_by_environment.index, y=priority_by_environment.values, palette='viridis')
plt.title('Median Priority Rating by Environment')
plt.xlabel('Environment')
plt.ylabel('Median Priority Rating')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
#Scatterplot to check correlation between priority and severity ratings
fig, ax = plt.subplots(figsize = (12, 6))
ax.scatter(bug_data["priority"], bug_data["severity"])
sns.regplot(x = bug_data['priority'], y = bug_data['severity'])
plt.suptitle('Correlation Between Severity and Priority', y=1.02)
plt.xlabel('Priority Rating')
plt.ylabel('Severity Rating')
plt.tight_layout()
plt.show()

#Check correlation coefficient
correlation = bug_data['priority'].corr(bug_data['severity'])
print("Correlation between priority and severity:", correlation)

In [ ]:
#Check for missing values
if bug_data.isna().values.any() == True:
  print("There are missing values in the dataset")
  print(bug_data.isna().sum())
else:
  print("There are no missing values in the dataset")

#Replace missing values with "Unknown"
bug_data_filled = bug_data.fillna("Unknown")

#Check
print(bug_data_filled.isna().sum())

In [ ]:
#Check for duplicate values
if bug_data_filled.duplicated().any() == True:
  print("There are duplicate values in the dataset")
  print(bug_data_filled.duplicated().sum())
else:
  print("There are no duplicate values in the dataset")

In [ ]:
#Combined severity/priority rating
bug_data_filled["severity_priority"] = (bug_data_filled["severity"] + bug_data_filled["priority"])//2